# Kafka Producer A: Random Number Stream

This notebook creates a simple Kafka producer for teaching stream joins.

It sends random numbers to the Kafka topic `teaching-stream-A`. Each message contains:

- `event_id`: a unique message ID
- `stream_id`: the source stream name, here `A`
- `number`: a random integer
- `event_time`: the time the event was created

Run this notebook first, then leave the final cell running while you start Producer B and the Spark join notebook.

In [ ]:
import json
import random
import time
import uuid
from datetime import datetime

from kafka3 import KafkaProducer

## Configuration

Change `HOST_IP` if your Kafka broker is running on a different machine.

For local Kafka, you can usually use:

```python
HOST_IP = "IP"
```

In [ ]:
HOST_IP = "10.192.8.146"
KAFKA_TOPIC = "teaching-stream-A"

producer = KafkaProducer(
    bootstrap_servers=[f"{HOST_IP}:9092"],
    value_serializer=lambda value: json.dumps(value).encode("utf-8"),
    api_version=(0, 10)
)

print(f"Producer A connected to Kafka topic: {KAFKA_TOPIC}")

## Stream Random Numbers

This cell continuously sends one random number every two seconds.

The random number range is deliberately small, from 1 to 5, so that Stream A and Stream B are likely to produce matching values. The Spark notebook will join the two streams when the same number appears in both streams within a short time window.

Stop this cell manually when you are finished.

In [ ]:
try:
    while True:
        event = {
            "event_id": str(uuid.uuid4()),
            "stream_id": "A",
            "number": random.randint(1, 5),
            "event_time": datetime.utcnow().isoformat()
        }

        producer.send(KAFKA_TOPIC, value=event)
        producer.flush()

        print(f"Sent from A: {event}")
        time.sleep(2)

except KeyboardInterrupt:
    print("Producer A stopped by user.")

finally:
    producer.close()
    print("Producer A connection closed.")